In [31]:
import json
import time
import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

In [32]:
file_path = 'data/Anovator_Links.xlsx'

# Loading the Excel file (with all the links already inside) into a DataFrame
df_links = pd.read_excel(file_path)

# Extracting the 'Links' column into a list
url_list = df_links['Links'].tolist()

print(f"Total links loaded: {len(url_list)}")
url_list[:5]

Total links loaded: 162


['https://www.anovator.com/report/index.html?id=480441268992984',
 'https://www.anovator.com/report/index.html?id=338208546978658',
 'https://www.anovator.com/report/index.html?id=220431535406836',
 'https://www.anovator.com/report/index.html?id=388571282933008',
 'https://www.anovator.com/report/index.html?id=188428370931439']

In [33]:
def scrape_anovator_report(url):
    try:
        # Extracting the ID from the URL
        report_id = url.split('id=')[-1].split('&')[0]
        
        # Internal API endpoint identified from the HTML source code
        api_url = f"https://www.anovator.com/BodyExamAction!getById.msg?id={report_id}"
        
        # Browser headers to prevent blocking
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "application/json, text/plain, */*",
            "Referer": url
        }

        # The website uses a POST request to fetch the report data
        response = requests.post(api_url, headers=headers, timeout=20)
        
        if response.status_code == 200:
            json_response = response.json()
            # The 'DATA' key contains the actual metrics 
            data = json_response.get('DATA', {})
            data['original_url'] = url
            return data
        else:
            return {"error": f"HTTP {response.status_code}", "original_url": url}
            
    except Exception as e:
        return {"error": str(e), "original_url": url}

# Testing on the first link to verify accuracy and see available fields
first_test_data = scrape_anovator_report(url_list[0])
first_test_data

{'aerobicGoal': 224.64,
 'age': 38,
 'anaGoal': 224.64,
 'armDim': 23.4979,
 'armSpan': 169.316,
 'bloodOxygen': 0,
 'bmi': 18.075975,
 'bmr': 1248.29,
 'bodyAge': 39,
 'bodyDetect': '{"bodyDirection":1,"fullBody":true,"humpbackRisk":5,"leftLegAngle":174.41129,"leftLegRisk":3,"left_ankle":{"score":0.85224503,"x":613.1382,"y":1105.1345},"left_ear":{"score":0.86975616,"x":602.6919,"y":238.09247},"left_elbow":{"score":0.8313005,"x":665.3696,"y":541.03485},"left_eye":{"score":0.8941862,"x":571.353,"y":217.19989},"left_hip":{"score":0.82636136,"x":613.1382,"y":676.8367},"left_knee":{"score":0.8535439,"x":602.6919,"y":896.2088},"left_mouth_corner":{"score":0.90048736,"x":571.353,"y":269.43134},"left_shoulder":{"score":0.8763466,"x":654.92334,"y":363.44794},"left_wrist":{"score":0.78968,"x":675.8159,"y":676.8367},"legRisk":3,"neck":{"score":0.8858804,"x":560.90674,"y":321.66278},"nose":{"score":0.9044601,"x":560.90674,"y":238.09247},"pelvisRisk":0,"rightLegAngle":180.1361,"rightLegRisk":1,"ri

In [34]:
all_reports = []

print(f"Number of reports to extract: {len(url_list)}")

# The loop is going through every URL in your list
for url in tqdm(url_list):
    # Calling the scraping function from the previous cell
    report_data = scrape_anovator_report(url)
    
    # Appending the result to the list
    all_reports.append(report_data)
    
    # Pausing for 1 second t(o avoid overwhelming the server)
    time.sleep(1)

# Converting the list of dictionaries into one large DataFrame
df_final = pd.DataFrame(all_reports)

print(f"Dataset shape: {df_final.shape}")
df_final.head()

Number of reports to extract: 162


100%|██████████| 162/162 [05:05<00:00,  1.89s/it]

Dataset shape: (162, 106)


,aerobicGoal,age,anaGoal,armDim,armSpan,bloodOxygen,bmi,bmr,bodyAge,bodyDetect,bodyImage,bodyImgOriginal,...,agility,balance,balanceAngle,customerId,rightVision,vitalCapacity,faceUserId,opinion,bloodMaxPressure,bloodMinPressure,leftVision,restingHeartRate
0,224.640,38,224.640,23.4979,169.316,0.0,18.075975,1248.29,39,"{""bodyDirection"":1,""fullBody"":true,""humpbackRi...",2025-08-14/547522127862180_ar.jpg,2025-08-14/228401716308181_ar.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,480.865,41,68.695,25.5552,163.257,0.0,22.958715,1249.99,42,"{""bodyDirection"":1,""fullBody"":true,""humpbackRi...",2025-09-25/455310796240977_ar.jpg,2025-09-25/386705250837914_ar.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,504.660,44,504.660,38.8264,176.823,0.0,29.824630,1941.01,43,"{""bodyDirection"":1,""fullBody"":true,""humpbackRi...",2025-09-25/547008894649424_ar.jpg,2025-09-25/650071169029631_ar.jpg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,482.820,37,482.820,36.3273,172.527,0.0,27.977335,1857.87,32,"{""bodyDirection"":1,""fullBody"":true,""humpbackRi...",2025-10-10/703582766700803_ar.jpg,2025-10-10/324801372602718_ar.jpg,...,443.0,4.0,80.7728,413261521817698,5.0,3553.0,NaN,NaN,NaN,NaN,NaN,NaN
4,363.090,30,484.120,35.1739,182.032,0.0,24.000723,1862.79,25,"{""bodyDirection"":1,""fullBody"":true,""humpbackRi...",2025-10-11/365121947439061_ar.jpg,2025-10-11/317261743357066_ar.jpg,...,464.0,3.0,60.8978,071267835471516,NaN,4233.0,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# General information about the data
print("Dataset Summary:")
print(df_final.info(verbose=False)) 
print()

# Checking for missing values in the entire dataset
missing_values = df_final.isnull().sum()
print("\nTop 10 columns with most missing values:")
print(missing_values.sort_values(ascending=False).head(10))
print()

# Inspecting key physical metrics (suggested from supervisor)
key_metrics = ['age', 'weight', 'bmi', 'fat', 'muscle', 'height', 'score']

# Only displaying columns that actually exist in the dataframe
available_metrics = [col for col in key_metrics if col in df_final.columns]

print("\nSample of key metrics from the first 10 reports:")
print(df_final[available_metrics].head(10))

# List of all available columns
print("\nAll captured column names:")
print(df_final.columns.tolist())

Dataset Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Columns: 106 entries, aerobicGoal to restingHeartRate
dtypes: float64(72), int64(10), object(24)
memory usage: 134.3+ KB
None


Top 10 columns with most missing values:
opinion             161
faceUserId          154
leftVision          153
rightVision         150
bloodMinPressure    149
bloodMaxPressure    149
restingHeartRate    144
sideImgOriginal     116
bodyImgOriginal     106
customerId          105
dtype: int64


Sample of key metrics from the first 10 reports:
   age  weight        bmi      fat   muscle  height  score
0   38    51.2  18.075975  20.8081  43.0824   168.3     75
1   41    60.7  22.958715  32.9260  36.8539   162.6     75
2   44    92.7  29.824630  19.9079  45.1264   176.3     88
3   37    82.0  27.977335  15.8555  48.5735   171.2     98
4   30    79.5  24.000723  13.2546  49.3580   182.0     98
5   16    60.9  23.611643  20.1240  44.3807   160.6     98
6   13    70.0  23.415750

In [36]:
# Converting timestamps from milliseconds to readable dates
df_final['date_created'] = pd.to_datetime(df_final['gmtCreate'], unit='ms')
df_final['date_modified'] = pd.to_datetime(df_final['gmtModify'], unit='ms')

# Moving important identification columns to the front
cols = df_final.columns.tolist()
first_cols = ['id', 'original_url', 'date_created', 'name', 'age', 'sex']
remaining_cols = [c for c in cols if c not in first_cols]

df_final = df_final[first_cols + remaining_cols]

# Dropping the old raw timestamp columns
df_final = df_final.drop(columns=['gmtCreate', 'gmtModify'])

df_final.head()

,id,original_url,date_created,name,age,sex,aerobicGoal,anaGoal,armDim,armSpan,bloodOxygen,bmi,...,balance,balanceAngle,customerId,rightVision,vitalCapacity,faceUserId,opinion,bloodMaxPressure,bloodMinPressure,leftVision,restingHeartRate,date_modified
0,480441268992984,https://www.anovator.com/report/index.html?id=...,2025-08-13 16:10:16,Unknown,38,F,224.640,224.640,23.4979,169.316,0.0,18.075975,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-08-13 16:15:26
1,338208546978658,https://www.anovator.com/report/index.html?id=...,2025-09-25 08:21:48,Unknown,41,F,480.865,68.695,25.5552,163.257,0.0,22.958715,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-25 08:28:15
2,220431535406836,https://www.anovator.com/report/index.html?id=...,2025-09-25 08:46:11,Unknown,44,M,504.660,504.660,38.8264,176.823,0.0,29.824630,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-25 08:50:10
3,388571282933008,https://www.anovator.com/report/index.html?id=...,2025-10-10 09:07:11,880224,37,M,482.820,482.820,36.3273,172.527,0.0,27.977335,...,4.0,80.7728,413261521817698,5.0,3553.0,NaN,NaN,NaN,NaN,NaN,NaN,2025-10-10 14:12:16
4,188428370931439,https://www.anovator.com/report/index.html?id=...,2025-10-10 11:56:17,950318,30,M,363.090,484.120,35.1739,182.032,0.0,24.000723,...,3.0,60.8978,071267835471516,NaN,4233.0,NaN,NaN,NaN,NaN,NaN,NaN,2025-10-11 06:45:12


In [37]:
# List of primary physical metrics for statistical analysis
numeric_metrics = [
    'age', 'weight', 'height', 'bmi', 'fat', 'muscle', 
    'score', 'protein', 'bone', 'water', 'waist', 'hips'
]

# Ensuring only stats for columns that actually exist in the df
valid_numeric_cols = [col for col in numeric_metrics if col in df_final.columns]

# Generating descriptive statistics
stats_summary = df_final[valid_numeric_cols].describe().transpose()

# Calculating the % of missing data per metric for quality verification
stats_summary['missing_pct'] = (df_final[valid_numeric_cols].isnull().sum() / len(df_final)) * 100

print("Physical Metrics Statistical Summary:")
stats_summary

Physical Metrics Statistical Summary:


,count,mean,std,min,25%,50%,75%,max,missing_pct
age,162.0,29.320988,12.195435,7.000000,19.000000,31.000000,37.000000,76.000000,0.0
weight,162.0,75.407407,19.272023,22.300000,63.600000,76.000000,87.950000,117.200000,0.0
height,162.0,174.553086,12.777689,124.000000,171.050000,176.700000,182.650000,199.500000,0.0
bmi,162.0,24.327269,4.498994,14.194401,21.354733,24.067351,26.648020,37.500893,0.0
fat,162.0,16.611112,6.999407,5.685170,12.457425,14.418100,20.575000,46.547500,0.0
muscle,162.0,46.719178,4.661300,29.953200,43.619325,47.950750,49.734425,56.455800,0.0
score,162.0,80.685185,15.427841,30.000000,73.250000,82.000000,98.000000,98.000000,0.0
protein,162.0,16.731762,1.773430,10.658600,15.457025,16.887500,18.114925,20.513000,0.0
bone,162.0,4.530072,1.413420,1.860000,3.741400,4.493020,5.050850,12.930000,0.0
water,162.0,62.062965,6.055707,38.887500,58.100000,62.939800,66.689400,72.100000,0.0


In [38]:
# Checking gender distribution
sex_distribution = df_final['sex'].value_counts()
print("Gender Distribution:")
print(sex_distribution)

# Checking for duplicated (In Report IDs)
# Ensuring each 'id' is unique in the final dataset
duplicate_count = df_final.duplicated(subset=['id']).sum()
print(f"\nNumber of duplicated Report IDs: {duplicate_count}")

# Checking for duplicated URLs (Logic: in case different IDs point to the same URL)
url_duplicate_count = df_final.duplicated(subset=['original_url']).sum()
print(f"Number of duplicated URLs: {url_duplicate_count}")

Gender Distribution:
sex
M    131
F     31
Name: count, dtype: int64

Number of duplicated Report IDs: 4
Number of duplicated URLs: 4


In [39]:
# Identifying the duplicated IDs
duplicate_ids = df_final[df_final.duplicated(subset=['id'], keep=False)]['id'].unique()

# Filtering the df to show all instances of these duplicated IDs
# Sorting by 'id' so the duplicates appear next to each other (for easy comparison)
df_duplicates = df_final[df_final['id'].isin(duplicate_ids)].sort_values(by='id')

print(f"Displaying all records for the {len(duplicate_ids)} duplicated IDs:")
df_duplicates[['id', 'name', 'age', 'weight', 'date_created']]

Displaying all records for the 4 duplicated IDs:


,id,name,age,weight,date_created
120,228844827788235,Unknown,25,76.0,2025-01-07 11:48:30
122,228844827788235,Unknown,25,76.0,2025-01-07 11:48:30
27,274672015638945,061202,18,71.8,2025-10-14 11:07:57
28,274672015638945,061202,18,71.8,2025-10-14 11:07:57
148,338671465758163,Unknown,36,107.7,2025-04-04 09:04:45
149,338671465758163,Unknown,36,107.7,2025-04-04 09:04:45
29,845042045098585,044311,22,88.9,2025-10-14 14:49:43
62,845042045098585,044311,22,88.9,2025-10-14 14:49:43


In [40]:
# Storing the original number of rows
original_row_count = len(df_final)

# Removing duplicates based on the 'id' column, keeping the first occurrence
df_final = df_final.drop_duplicates(subset=['id'], keep='first')

# Calculating how many rows were removed
removed_count = original_row_count - len(df_final)

print(f"Removed {removed_count} duplicate records.")
print(f"New dataset shape: {df_final.shape}")

# Verifying no duplicate IDs remain
remaining_duplicates = df_final.duplicated(subset=['id']).sum()
print(f"Remaining duplicated IDs: {remaining_duplicates}")

Removed 4 duplicate records.
New dataset shape: (158, 106)
Remaining duplicated IDs: 0


In [41]:

# Function to extract specific angles from the nested bodyDetect JSON string
def extract_front_angles(json_str):
    try:
        if pd.isna(json_str) or json_str == "":
            return None, None, None
        data = json.loads(json_str)
        return data.get('shoulderAngle'), data.get('leftLegAngle'), data.get('rightLegAngle')
    except:
        return None, None, None

# Applying extraction and creating new columns
df_final[['front_shoulder_angle', 'front_left_leg_angle', 'front_right_leg_angle']] = df_final['bodyDetect'].apply(
    lambda x: pd.Series(extract_front_angles(x))
)

print("Extracted joint angles from the front-view body detection data")
df_final[['id', 'front_shoulder_angle', 'front_left_leg_angle', 'front_right_leg_angle']].head()

Extracted joint angles from the front-view body detection data


,id,front_shoulder_angle,front_left_leg_angle,front_right_leg_angle
0,480441268992984,3.012788,174.41129,180.13610
1,338208546978658,0.000000,180.57727,179.86390
2,220431535406836,0.000000,182.86240,177.13759
3,388571282933008,2.726318,188.30275,169.00750
4,188428370931439,0.000000,179.54648,177.27368


In [42]:
# Function to extract side-view metrics from the nested sideBodyDetect JSON string
def extract_side_metrics(json_str):
    try:
        if pd.isna(json_str) or json_str == "":
            return None, None, None
        data = json.loads(json_str)
        # Extracting specific side-view measurements
        return data.get('humpbackRisk'), data.get('leftLegAngle'), data.get('rightLegAngle')
    except:
        return None, None, None

# Applying extraction and creating new columns for side-view metrics
df_final[['side_hunchback_risk', 'side_left_leg_angle', 'side_right_leg_angle']] = df_final['sideBodyDetect'].apply(
    lambda x: pd.Series(extract_side_metrics(x))
)

print("Extracted metrics from the side-view body detection data")
df_final[['id', 'side_hunchback_risk', 'side_left_leg_angle', 'side_right_leg_angle']].head()

Extracted metrics from the side-view body detection data


,id,side_hunchback_risk,side_left_leg_angle,side_right_leg_angle
0,480441268992984,1.0,191.30994,185.44035
1,338208546978658,4.0,175.03026,174.96115
2,220431535406836,4.0,168.69006,171.02737
3,388571282933008,1.0,175.03027,177.01573
4,188428370931439,1.0,174.19554,179.54650


In [43]:
new_postural_cols = [
    'front_shoulder_angle', 'front_left_leg_angle', 'front_right_leg_angle',
    'side_hunchback_risk', 'side_left_leg_angle', 'side_right_leg_angle'
]

# Generating descriptive statistics for the new columns
postural_summary = df_final[new_postural_cols].describe().transpose()

# Adding a column to show how many values are missing (NaN)
postural_summary['missing_count'] = df_final[new_postural_cols].isnull().sum()

print("Postural Metrics Investigation Summary:")
postural_summary

Postural Metrics Investigation Summary:


,count,mean,std,min,25%,50%,75%,max,missing_count
front_shoulder_angle,111.0,1.019653,1.558165,0.00000,0.00000,0.00000,2.726313,6.709827,47
front_left_leg_angle,111.0,181.478366,3.419537,174.26090,179.54648,182.60255,183.096310,190.627580,47
front_right_leg_angle,111.0,178.118922,4.003729,167.93512,174.55967,177.39743,180.129925,188.972630,47
side_hunchback_risk,103.0,2.281553,1.166673,1.00000,1.00000,2.00000,3.000000,5.000000,55
side_left_leg_angle,103.0,179.536832,4.704711,168.69006,177.01821,180.00000,182.782045,191.421200,55
side_right_leg_angle,103.0,179.833093,4.709291,168.69006,177.00147,179.86389,182.714040,188.972640,55


In [44]:
# Calculating the difference b/en actual age and body age
# A negative value means the person is "biologically younger" than their actual age (Assumption)
df_final['age_gap'] = df_final['bodyAge'] - df_final['age']

# Inspecting the relationship b/en Score, Body Age, and Body Shape
health_focus_cols = ['id', 'age', 'bodyAge', 'age_gap', 'score', 'bodyShape']

print("Age Gap Statistics (Body Age - Actual Age):")
print(df_final['age_gap'].describe())

print("\nSample of Health Index data:")
df_final[health_focus_cols].head(10)

Age Gap Statistics (Body Age - Actual Age):
count    158.000000
mean       0.449367
std        3.303246
min       -5.000000
25%       -1.000000
50%        0.000000
75%        1.000000
max        8.000000
Name: age_gap, dtype: float64

Sample of Health Index data:


,id,age,bodyAge,age_gap,score,bodyShape
0,480441268992984,38,39,1,75,4
1,338208546978658,41,42,1,75,0
2,220431535406836,44,43,-1,88,5
3,388571282933008,37,32,-5,98,5
4,188428370931439,30,25,-5,98,7
5,587378513608375,16,16,0,98,7
6,432421340658149,13,14,1,98,7
7,606612128598693,15,16,1,98,3
8,812457621265118,15,16,1,98,3
9,674638527002663,15,15,0,98,6


In [45]:
# Grouping by bodyShape to check the count and avg health score
body_shape_analysis = df_final.groupby('bodyShape').agg({
    'id': 'count',
    'score': 'mean',
    'fat': 'mean',
    'muscle': 'mean'
}).rename(columns={'id': 'count'})

# Sorting by count to see the most common body shapes
body_shape_analysis = body_shape_analysis.sort_values(by='count', ascending=False)

print("Analysis of Body Shape Categories:")
body_shape_analysis

Analysis of Body Shape Categories:


,count,score,fat,muscle
bodyShape,,,,
7,49,85.979592,14.652810,48.090757
3,26,85.769231,11.718047,49.805685
4,23,85.782609,17.489909,45.456952
1,21,66.952381,24.396538,42.001519
5,16,82.000000,17.676306,46.194475
6,15,78.133333,8.760010,51.336100
2,5,37.200000,35.960200,35.522060
0,3,76.666667,32.276133,37.531233


In [46]:
# Selecting the most important metrics for correlation
correlation_cols = [
    'score', 'age', 'weight', 'height', 'bmi', 'fat', 
    'muscle', 'water', 'protein', 'bone', 'age_gap'
]

# Calculating the correlation matrix
    #  1.0 is perfect positive correlation
    # -1.0 is perfect negative correlation
corr_matrix = df_final[correlation_cols].corr()

# Focusing specifically on how everything correlates with the 'score'
score_correlations = corr_matrix['score'].sort_values(ascending=False)

print("Correlation of metrics with Health Score:")
print(score_correlations)

Correlation of metrics with Health Score:
score      1.000000
water      0.491117
muscle     0.473112
protein    0.408840
height     0.268820
bone      -0.006112
weight    -0.151970
age       -0.242095
bmi       -0.333592
fat       -0.463461
age_gap   -0.776387
Name: score, dtype: float64


In [47]:
# List of columns that represent postural risk assessments
risk_columns = [
    'humpbackRisk', 'spineRisk', 'pelvisRisk', 
    'postureRisk', 'kneeRisk', 'frontHeadRisk', 'side_hunchback_risk'
]

# Creating a dictionary to store the counts for each risk level
risk_summary = {}

for col in risk_columns:
    if col in df_final.columns:
        # Getting the distribution of values, ignoring NaN
        risk_summary[col] = df_final[col].value_counts().sort_index()

print(" Frequency of Risk Levels across categories:")
print("(Rows are risk levels; Columns are categories)")
pd.DataFrame(risk_summary).fillna(0).astype(int)

 Frequency of Risk Levels across categories:
(Rows are risk levels; Columns are categories)


,humpbackRisk,spineRisk,pelvisRisk,postureRisk,kneeRisk,frontHeadRisk,side_hunchback_risk
-2.0,0,0,3,0,0,0,0
-1.0,0,0,4,47,0,0,0
1.0,36,108,12,3,101,32,36
2.0,21,3,13,58,0,8,21
3.0,31,0,30,50,0,0,31
4.0,11,0,17,0,0,9,11
5.0,4,0,24,0,2,54,4


In [48]:
# Creating columns to check the sum of segments against the reported totals
fat_columns = ['fatLeftArm', 'fatRightArm', 'fatLeftLeg', 'fatRightLeg', 'fatTrunk']
muscle_columns = ['muscleLeftArm', 'muscleRightArm', 'muscleLeftLeg', 'muscleRightLeg', 'muscleTrunk']

# Calculating Sums
df_final['calc_total_fat'] = df_final[fat_columns].sum(axis=1)
df_final['calc_total_muscle'] = df_final[muscle_columns].sum(axis=1)

# Calculating the Differences (absolute error)
df_final['fat_error'] = (df_final['fat'] - df_final['calc_total_fat']).abs()
df_final['muscle_error'] = (df_final['muscle'] - df_final['calc_total_muscle']).abs()

print("Data Integrity Check (Segmental Sum vs Total):")
print(f"Average Fat discrepancy: {df_final['fat_error'].mean():.6f} kg")
print(f"Average Muscle discrepancy: {df_final['muscle_error'].mean():.6f} kg")

print("\nSample of Segmental Muscle Breakdown (kg):")
df_final[['id', 'muscle', 'calc_total_muscle', 'muscleTrunk', 'muscleLeftArm', 'muscleRightArm', 'muscleLeftLeg', 'muscleRightLeg']].head()

Data Integrity Check (Segmental Sum vs Total):
Average Fat discrepancy: 6.794300 kg
Average Muscle discrepancy: 13.421608 kg

Sample of Segmental Muscle Breakdown (kg):


,id,muscle,calc_total_muscle,muscleTrunk,muscleLeftArm,muscleRightArm,muscleLeftLeg,muscleRightLeg
0,480441268992984,43.0824,32.62616,16.5460,1.62307,1.70225,6.32766,6.42718
1,338208546978658,36.8539,34.70006,18.4451,1.97661,1.99517,6.18741,6.09577
2,220431535406836,45.1264,63.94324,33.7444,4.65218,4.71826,10.23540,10.59300
3,388571282933008,48.5735,59.92150,31.9326,4.38503,4.32778,9.59210,9.68399
4,188428370931439,49.3580,61.45511,31.4470,4.08623,4.18208,10.85510,10.88470


In [49]:
# Defining the output path
final_output_path = 'data/Anovator_Final_Cleaned_Dataset.xlsx'

# Saving the final dataframe to Excel
df_final.to_excel(final_output_path, index=False)

print(f"Final dataset successfully saved to: {final_output_path}")
print(f"Total records saved: {len(df_final)}")
print(f"Total columns saved: {len(df_final.columns)}")

Final dataset successfully saved to: data/Anovator_Final_Cleaned_Dataset.xlsx
Total records saved: 158
Total columns saved: 117


## EDA

In [50]:
# DUPLICATE CHECK
duplicate_ids = df_final.duplicated(subset=['id']).sum()
duplicate_rows = df_final.duplicated().sum()

# MISSING DATA ANALYSIS
# Calculating percentage of missing values per column
missing_pct = df_final.isnull().mean() * 100

# Identifying columns that are entirely empty
empty_cols = missing_pct[missing_pct == 100].index.tolist()
# Identifying columns with more than 50% missing data
high_missing_cols = missing_pct[missing_pct > 50].index.tolist()


# STATISTICAL OUTLIER DETECTION (Range Check)
# Checking for potentially unusual values in core metrics
outlier_check = df_final[['age', 'weight', 'height', 'bmi', 'fat', 'muscle']].describe()


# DATA CONSISTENCY FLAGS
# Flagging rows, where the limb sum significantly differs from the reported total (> 1kg difference --> Assumption)
fat_sum_issue = (df_final['fat_error'] > 1.0).sum()
muscle_sum_issue = (df_final['muscle_error'] > 1.0).sum()

# PRINTING THE REPORT
print("DATA REPORT")
print(f"Total Records: {len(df_final)}")
print(f"Total Columns: {len(df_final.columns)}")
print(f"Duplicated IDs found: {duplicate_ids}")
print(f"Completely identical rows: {duplicate_rows}")

print("\nMissing Data Metrics")
print(f"Columns that are 100% empty: {len(empty_cols)}")
if empty_cols: print(f"List: {empty_cols}")
print(f"Columns with > 50% missing data: {len(high_missing_cols)}")
print(f"Sample of high-missing columns: {high_missing_cols[:10]}")

print("\nUnusual Physical Ranges")
print(outlier_check)

print("\nInternal Consistency Issues")
print(f"Records where limb fat sum does NOT match total fat: {fat_sum_issue}")
print(f"Records where limb muscle sum does NOT match total muscle: {muscle_sum_issue}")
print("Note: Significant discrepancies here usually indicate the machine uses different medical formulas for totals vs. segments.")

print("\nUnique Values in Identification ")
print(f"Unique names: {df_final['name'].nunique()}")
print(f"Unique gym IDs: {df_final['gymId'].nunique()}")
print(f"Unique language settings: {df_final['language'].unique().tolist()}")

print("\nFinal Column Data Types")
print(df_final.dtypes.value_counts())

DATA REPORT
Total Records: 158
Total Columns: 117
Duplicated IDs found: 0
Completely identical rows: 0

Missing Data Metrics
Columns that are 100% empty: 0
Columns with > 50% missing data: 10
Sample of high-missing columns: ['bodyImgOriginal', 'sideImgOriginal', 'customerId', 'rightVision', 'faceUserId', 'opinion', 'bloodMaxPressure', 'bloodMinPressure', 'leftVision', 'restingHeartRate']

Unusual Physical Ranges
              age      weight      height         bmi         fat      muscle
count  158.000000  158.000000  158.000000  158.000000  158.000000  158.000000
mean    29.424051   75.136709  174.352532   24.290908   16.633561   46.698062
std     12.286026   19.310956   12.860514    4.516909    7.032627    4.686866
min      7.000000   22.300000  124.000000   14.194401    5.685170   29.953200
25%     19.000000   63.525000  170.850000   21.327884   12.501350   43.619325
50%     31.000000   75.950000  176.600000   24.016382   14.514100   47.922750
75%     37.000000   86.750000  182.475

In [51]:
# List of columns to drop based on high missing values (>50%)
cols_to_drop_missing = [
    'bodyImgOriginal', 'sideImgOriginal', 'customerId', 'rightVision', 
    'faceUserId', 'opinion', 'bloodMaxPressure', 'bloodMinPressure', 
    'leftVision', 'restingHeartRate'
]

# List of technical/metadata columns that provide no health value for a model
cols_to_drop_technical = [
    'original_url', 'gymId', 'gymName', 'ip', 'language', 'dataVersion', 
    'deviceFrom', 'isDeleted', 'unitType', 'gmtCreate', 'gmtModify',
    'bodyImage', 'sideImage', 'fat_error', 'muscle_error'
]

# Combining the lists and drop only if they exist in the dataframe
final_drop_list = list(set(cols_to_drop_missing + cols_to_drop_technical))
df_model_ready = df_final.drop(columns=[col for col in final_drop_list if col in df_final.columns])

print(f"Removed {len(final_drop_list)} non-predictive or empty columns.")
print(f"Remaining columns for modeling: {len(df_model_ready.columns)}")
df_model_ready.head()

Removed 25 non-predictive or empty columns.
Remaining columns for modeling: 94


,id,date_created,name,age,sex,aerobicGoal,anaGoal,armDim,armSpan,bloodOxygen,bmi,bmr,...,balanceAngle,vitalCapacity,date_modified,front_shoulder_angle,front_left_leg_angle,front_right_leg_angle,side_hunchback_risk,side_left_leg_angle,side_right_leg_angle,age_gap,calc_total_fat,calc_total_muscle
0,480441268992984,2025-08-13 16:10:16,Unknown,38,F,224.640,224.640,23.4979,169.316,0.0,18.075975,1248.29,...,NaN,NaN,2025-08-13 16:15:26,3.012788,174.41129,180.13610,1.0,191.30994,185.44035,1,9.676550,32.62616
1,338208546978658,2025-09-25 08:21:48,Unknown,41,F,480.865,68.695,25.5552,163.257,0.0,22.958715,1249.99,...,NaN,NaN,2025-09-25 08:28:15,0.000000,180.57727,179.86390,4.0,175.03026,174.96115,1,18.544620,34.70006
2,220431535406836,2025-09-25 08:46:11,Unknown,44,M,504.660,504.660,38.8264,176.823,0.0,29.824630,1941.01,...,NaN,NaN,2025-09-25 08:50:10,0.000000,182.86240,177.13759,4.0,168.69006,171.02737,-1,16.833397,63.94324
3,388571282933008,2025-10-10 09:07:11,880224,37,M,482.820,482.820,36.3273,172.527,0.0,27.977335,1857.87,...,80.7728,3553.0,2025-10-10 14:12:16,2.726318,188.30275,169.00750,1.0,175.03027,177.01573,-5,11.280844,59.92150
4,188428370931439,2025-10-10 11:56:17,950318,30,M,363.090,484.120,35.1739,182.032,0.0,24.000723,1862.79,...,60.8978,4233.0,2025-10-11 06:45:12,0.000000,179.54648,177.27368,1.0,174.19554,179.54650,-5,8.232662,61.45511


In [52]:
# Creating a copy for the modeling process
df_ml = df_model_ready.copy()

# Converting 'sex' to numeric: 
    # M = 1 
    # F = 0
df_ml['sex'] = df_ml['sex'].map({'M': 1, 'F': 0})

# Dropping columns that are names, IDs, or Dates
# These are identifiers and will cause the model to "overfit" (memorize individuals)
cols_to_drop_final = ['id', 'name', 'date_created', 'date_modified']
df_ml = df_ml.drop(columns=[col for col in cols_to_drop_final if col in df_ml.columns])

# Handling the 'bodyDetect' and 'sideBodyDetect' raw JSON columns
df_ml = df_ml.drop(columns=['bodyDetect', 'sideBodyDetect'], errors='ignore')

# Filtering for only numeric columns to ensure the model doesn't crash
df_ml = df_ml.select_dtypes(include=['number'])

print(f"Final feature count for modeling: {len(df_ml.columns)}")
print("\nRemaining columns in the ML dataset:")
print(df_ml.columns.tolist())

Final feature count for modeling: 85

Remaining columns in the ML dataset:
['age', 'sex', 'aerobicGoal', 'anaGoal', 'armDim', 'armSpan', 'bloodOxygen', 'bmi', 'bmr', 'bodyAge', 'bodyShape', 'bodyShapeRisk', 'bone', 'caloriesInput', 'enduGoal', 'fat', 'fatControl', 'fatLeftArm', 'fatLeftLeg', 'fatRightArm', 'fatRightLeg', 'fatTrunk', 'footLength', 'frontHeadRisk', 'head', 'headRisk', 'heartFun', 'height', 'hips', 'humpbackRisk', 'inFat', 'kneeRisk', 'lowerBody', 'muscle', 'muscleControl', 'muscleLeftArm', 'muscleLeftLeg', 'muscleRightArm', 'muscleRightLeg', 'muscleTrunk', 'occipitalSpace', 'pelvisRisk', 'perfectWeight', 'postureRisk', 'protein', 'r100LeftArm', 'r100LeftLeg', 'r100RightArm', 'r100RightLeg', 'r100Trunk', 'r20LeftArm', 'r20LeftLeg', 'r20RightArm', 'r20RightLeg', 'r20Trunk', 'score', 'shank', 'shoulderRisk', 'shoulderWidth', 'spineRisk', 'sportGoal', 'sportLevel', 'sportSafeRisk', 'subFat', 'thigh', 'thighDim', 'upperBody', 'waist', 'water', 'wc', 'weight', 'weightControl',

In [53]:
# Identifying columns that still have missing values
missing_counts = df_ml.isnull().sum()
cols_to_fix = missing_counts[missing_counts > 0]

print("Columns with missing values before imputation:")
print(cols_to_fix)

# Filling missing values with the Median of their respective columns
# Median > Mean for health data to avoid distortion (from extreme athletes or children)
df_ml = df_ml.fillna(df_ml.median())

remaining_missing = df_ml.isnull().sum().sum()
print(f"\nTotal missing values remaining: {remaining_missing}")

Columns with missing values before imputation:
armSpan                  47
bloodOxygen              27
footLength               55
frontHeadRisk            55
head                     47
headRisk                 47
heartFun                 27
humpbackRisk             55
kneeRisk                 55
lowerBody                47
occipitalSpace           47
pelvisRisk               55
shank                    47
shoulderRisk             47
shoulderWidth            47
spineRisk                47
thigh                    47
upperBody                47
agility                  62
balance                  59
balanceAngle             59
vitalCapacity            69
front_shoulder_angle     47
front_left_leg_angle     47
front_right_leg_angle    47
side_hunchback_risk      55
side_left_leg_angle      55
side_right_leg_angle     55
dtype: int64

Total missing values remaining: 0


In [54]:
df_ml.head()

,age,sex,aerobicGoal,anaGoal,armDim,armSpan,bloodOxygen,bmi,bmr,bodyAge,bodyShape,bodyShapeRisk,...,balance,balanceAngle,vitalCapacity,front_shoulder_angle,front_left_leg_angle,front_right_leg_angle,side_hunchback_risk,side_left_leg_angle,side_right_leg_angle,age_gap,calc_total_fat,calc_total_muscle
0,38,0,224.640,224.640,23.4979,169.316,0.0,18.075975,1248.29,39,4,2,...,3.0,70.4404,3910.0,3.012788,174.41129,180.13610,1.0,191.30994,185.44035,1,9.676550,32.62616
1,41,0,480.865,68.695,25.5552,163.257,0.0,22.958715,1249.99,42,0,3,...,3.0,70.4404,3910.0,0.000000,180.57727,179.86390,4.0,175.03026,174.96115,1,18.544620,34.70006
2,44,1,504.660,504.660,38.8264,176.823,0.0,29.824630,1941.01,43,5,2,...,3.0,70.4404,3910.0,0.000000,182.86240,177.13759,4.0,168.69006,171.02737,-1,16.833397,63.94324
3,37,1,482.820,482.820,36.3273,172.527,0.0,27.977335,1857.87,32,5,2,...,4.0,80.7728,3553.0,2.726318,188.30275,169.00750,1.0,175.03027,177.01573,-5,11.280844,59.92150
4,30,1,363.090,484.120,35.1739,182.032,0.0,24.000723,1862.79,25,7,2,...,3.0,60.8978,4233.0,0.000000,179.54648,177.27368,1.0,174.19554,179.54650,-5,8.232662,61.45511


In [55]:
# Creating a dictionary to map columns to their physical units
unit_map = {
    # Mass / Weight (kg)
    'kg': ['weight', 'fat', 'muscle', 'bone', 'protein', 'water', 'inFat', 'subFat', 'weightControl', 
           'fatControl', 'muscleControl', 'perfectWeight', 'calc_total_fat', 'calc_total_muscle'] + 
          [col for col in df_ml.columns if 'fat' in col.lower() or 'muscle' in col.lower() if col not in ['fat', 'muscle']],
    
    # Length / Circumference (cm)
    'cm': ['height', 'armDim', 'armSpan', 'head', 'hips', 'lowerBody', 'upperBody', 'shank', 
           'shoulderWidth', 'thigh', 'thighDim', 'waist', 'occipitalSpace', 'footLength'],
    
    # Energy (kcal)
    'kcal': ['bmr', 'caloriesInput', 'aerobicGoal', 'anaGoal', 'enduGoal', 'sportGoal'],
    
    # Time (ms)
    'ms': ['agility'],
    
    # Volume (ml)
    'ml': ['vitalCapacity'],
    
    # Electrical Impedance (Ohms Ω)
    'ohm': [col for col in df_ml.columns if col.startswith('r100') or col.startswith('r20')],
    
    # Angles (Degrees °)
    'deg': ['front_shoulder_angle', 'front_left_leg_angle', 'front_right_leg_angle', 
            'side_left_leg_angle', 'side_right_leg_angle', 'balanceAngle'],
    
    # Unitless / Ratios / Scores
    'score_or_ratio': ['score', 'bmi', 'wc', 'bodyShape', 'bodyAge', 'age_gap', 'sportLevel'],
    
    # Risk Levels (Scale 1-5)
    'risk_level': [col for col in df_ml.columns if 'Risk' in col or 'risk' in col]
}

# Creating a flat list for easy reading
unit_report = []
for unit, columns in unit_map.items():
    for col in columns:
        if col in df_ml.columns:
            unit_report.append({'Column': col, 'Unit': unit})

df_units = pd.DataFrame(unit_report)

# Verifying BMI calculation manually for row 0 to confirm units: Weight(kg) / [Height(m)]^2
row0 = df_ml.iloc[0]
manual_bmi = row0['weight'] / ((row0['height']/100)**2)
print(f"Manual BMI Check: {manual_bmi:.2f} | Machine BMI: {row0['bmi']:.2f}")
print("\nFirst 20 column units identified:")
df_units.head(40)

Manual BMI Check: 18.08 | Machine BMI: 18.08

First 20 column units identified:


,Column,Unit
0,weight,kg
1,fat,kg
2,muscle,kg
3,bone,kg
4,protein,kg
5,water,kg
6,inFat,kg
7,subFat,kg
8,weightControl,kg
9,fatControl,kg


In [56]:
# Comprehensive mapping of English keys to Bulgarian translations
# AI Generated:

bulgarian_translation = {
    'age': 'Възраст', 'sex': 'Пол (1=М, 0=Ж)', 'weight': 'Тегло', 'height': 'Височина',
    'bmi': 'ИТМ (Индекс на телесна маса)', 'bmr': 'БМР (Базален метаболизъм)',
    'fat': 'Мазнини (Общо)', 'muscle': 'Мускулна маса (Общо)', 'water': 'Вода',
    'protein': 'Протеини', 'bone': 'Костна маса', 'inFat': 'Висцерални мазнини',
    'subFat': 'Подкожни мазнини', 'waist': 'Талия', 'hips': 'Ханш',
    'wc': 'WHR (Съотношение талия-ханш)', 'score': 'Здравен резултат',
    'bodyAge': 'Телесна възраст', 'age_gap': 'Разлика във възрастта',
    'bodyShape': 'Форма на тялото', 'bodyShapeRisk': 'Риск от формата на тялото',
    'weightControl': 'Контрол на теглото', 'fatControl': 'Контрол на мазнините',
    'muscleControl': 'Контрол на мускулите', 'perfectWeight': 'Идеално тегло',
    'armDim': 'Обиколка на ръката', 'armSpan': 'Размах на ръцете',
    'head': 'Обиколка на главата', 'footLength': 'Дължина на стъпалото',
    'lowerBody': 'Долна част на тялото (дължина)', 'upperBody': 'Горна част на тялото (дължина)',
    'shank': 'Подбедрица', 'thigh': 'Бедро (дължина)', 'thighDim': 'Обиколка на бедрото',
    'shoulderWidth': 'Ширина на раменете', 'occipitalSpace': 'Разстояние до тила',
    'aerobicGoal': 'Аеробна цел', 'anaGoal': 'Анаеробна цел',
    'enduGoal': 'Цел за издръжливост', 'sportGoal': 'Спортна цел',
    'caloriesInput': 'Прием на калории', 'agility': 'Пъргавина',
    'vitalCapacity': 'Витален капацитет', 'balance': 'Баланс',
    'balanceAngle': 'Ъгъл на баланс', 'humpbackRisk': 'Риск от гърбица (Преден)',
    'spineRisk': 'Риск за гръбначния стълб', 'pelvisRisk': 'Риск за таза',
    'postureRisk': 'Риск за стойката', 'kneeRisk': 'Риск за коленете',
    'frontHeadRisk': 'Риск за позиция на главата', 'front_shoulder_angle': 'Преден ъгъл на рамото',
    'front_left_leg_angle': 'Преден ъгъл на ляв крак', 'front_right_leg_angle': 'Преден ъгъл на десен крак',
    'side_hunchback_risk': 'Риск от гърбица (Страничен)', 'side_left_leg_angle': 'Страничен ъгъл на ляв крак',
    'side_right_leg_angle': 'Страничен ъгъл на десен крак', 'calc_total_fat': 'Изчислени мазнини (сума)',
    'calc_total_muscle': 'Изчислени мускули (сума)'
}

# Creating a list for the Excel output
review_data = []
for col in df_ml.columns:
    # Getting Bulgarian name if exists, otherwise keep English
    bg_name = bulgarian_translation.get(col, col)
    
    # Identifying units
    unit = 'N/A'
    for u, cols in unit_map.items():
        if col in cols:
            unit = u
            break
            
    review_data.append({
        'Original_Column_Name': col,
        'Bulgarian_Translation': bg_name,
        'Identified_Unit': unit
    })

# Converting to df and saving
df_review = pd.DataFrame(review_data)
review_output_path = 'data/Column_Units_Review_Bulgarian.xlsx'
df_review.to_excel(review_output_path, index=False)

print(f"Review file created: {review_output_path}")
df_review.head(15)

Review file created: data/Column_Units_Review_Bulgarian.xlsx


,Original_Column_Name,Bulgarian_Translation,Identified_Unit
0,age,Възраст,N/A
1,sex,"Пол (1=М, 0=Ж)",N/A
2,aerobicGoal,Аеробна цел,kcal
3,anaGoal,Анаеробна цел,kcal
4,armDim,Обиколка на ръката,cm
5,armSpan,Размах на ръцете,cm
6,bloodOxygen,bloodOxygen,N/A
7,bmi,ИТМ (Индекс на телесна маса),score_or_ratio
8,bmr,БМР (Базален метаболизъм),kcal
9,bodyAge,Телесна възраст,score_or_ratio


In [57]:
# Map of units from English to Bulgarian

unit_map_bg = {
    'kg': 'кг',
    'cm': 'см',
    'kcal': 'ккал',
    'ms': 'мс',
    'ml': 'мл',
    'ohm': 'ом',
    'deg': 'градуси',
    'score_or_ratio': 'резултат/съотношение',
    'risk_level': 'степен на риск (1-5)',
    'N/A': 'няма'
}

# Mapping for limbs to Bulgarian for segmental columns
limb_map_bg = {
    'LeftArm': 'лява ръка',
    'RightArm': 'дясна ръка',
    'LeftLeg': 'ляв крак',
    'RightLeg': 'десен крак',
    'Trunk': 'торс'
}

final_review_list = []

for col in df_ml.columns:
    # Determining Bulgarian Name
    bg_name = bulgarian_translation.get(col)
    
    # Handling segmental columns automatically if not in the dictionary
    if not bg_name:
        if 'fat' in col:
            limb = col.replace('fat', '')
            bg_name = f"Мазнини ({limb_map_bg.get(limb, limb)})"
        elif 'muscle' in col:
            limb = col.replace('muscle', '')
            bg_name = f"Мускулна маса ({limb_map_bg.get(limb, limb)})"
        elif col.startswith('r100'):
            limb = col.replace('r100', '')
            bg_name = f"Импеданс 100kHz ({limb_map_bg.get(limb, limb)})"
        elif col.startswith('r20'):
            limb = col.replace('r20', '')
            bg_name = f"Импеданс 20kHz ({limb_map_bg.get(limb, limb)})"
        else:
            bg_name = col # Fallback to English if no match

    # Determining Unit in Bulgarian
    eng_unit = 'N/A'
    for u, columns in unit_map.items():
        if col in columns:
            eng_unit = u
            break
    bg_unit = unit_map_bg.get(eng_unit, 'няма')

    final_review_list.append({
        'Име на колоната': bg_name,
        'Мерна единица': bg_unit
    })

# Creating DataFrame and Export
df_bg_review = pd.DataFrame(final_review_list)
bg_review_path = 'data/Prehled_Koloni_i_Edinici.xlsx'
df_bg_review.to_excel(bg_review_path, index=False)

print(f"File created: {bg_review_path}")
print("This file contains only Bulgarian descriptions and Bulgarian units")
df_bg_review.head(20)

File created: data/Prehled_Koloni_i_Edinici.xlsx
This file contains only Bulgarian descriptions and Bulgarian units


,Име на колоната,Мерна единица
0,Възраст,няма
1,"Пол (1=М, 0=Ж)",няма
2,Аеробна цел,ккал
3,Анаеробна цел,ккал
4,Обиколка на ръката,см
5,Размах на ръцете,см
6,bloodOxygen,няма
7,ИТМ (Индекс на телесна маса),резултат/съотношение
8,БМР (Базален метаболизъм),ккал
9,Телесна възраст,резултат/съотношение


In [58]:
# Setting pandas display options to ensure = enough columns to be seen
pd.set_option('display.max_columns', 25)

# Dataset Scale Overview
print(f"Total Unique Records: {len(df_final)}")
print(f"Total Columns: {len(df_final.columns)}")

# Displaying core identification and physical health metrics
# Selecting these specific columns to keep the table readable 
display_cols = [
    'id', 'name', 'age', 'sex', 'weight', 'height', 'bmi', 'score',
    'fat', 'muscle', 'date_created', 'front_shoulder_angle', 'side_hunchback_risk'
]

print("\nDetailed sample of core metrics (First 5 rows):")
print(df_final[display_cols].head())

# Statistical Range Check
# Showing the minimum, maximum, and avg for the most critical modeling features
print("\nStatistical ranges for physical metrics:")
summary_metrics = ['age', 'weight', 'height', 'bmi', 'score', 'fat', 'muscle', 'water']
print(df_final[summary_metrics].agg(['min', 'max', 'mean']).round(2))

# 4. Identification of the target
print(f"\nTarget variable for future modeling: 'score'")

Total Unique Records: 158
Total Columns: 117

Detailed sample of core metrics (First 5 rows):
                id     name  age sex  weight  height        bmi  score  \
0  480441268992984  Unknown   38   F    51.2   168.3  18.075975     75   
1  338208546978658  Unknown   41   F    60.7   162.6  22.958715     75   
2  220431535406836  Unknown   44   M    92.7   176.3  29.824630     88   
3  388571282933008   880224   37   M    82.0   171.2  27.977335     98   
4  188428370931439   950318   30   M    79.5   182.0  24.000723     98   

       fat   muscle        date_created  front_shoulder_angle  \
0  20.8081  43.0824 2025-08-13 16:10:16              3.012788   
1  32.9260  36.8539 2025-09-25 08:21:48              0.000000   
2  19.9079  45.1264 2025-09-25 08:46:11              0.000000   
3  15.8555  48.5735 2025-10-10 09:07:11              2.726318   
4  13.2546  49.3580 2025-10-10 11:56:17              0.000000   

   side_hunchback_risk  
0                  1.0  
1                  4

In [59]:
# Defining the path for the processed dataset
model_data_path = 'data/Anovator_Processed_Modeling_Data.xlsx'

# Saving the numeric-only dataset (df_ml) to Excel
# This version has 85 columns and is what will be used for the model
df_ml.to_excel(model_data_path, index=False)

print(f"Processed dataset saved successfully to: {model_data_path}")
print(f"Total columns in this file: {len(df_ml.columns)}")

Processed dataset saved successfully to: data/Anovator_Processed_Modeling_Data.xlsx
Total columns in this file: 85


In [60]:
# Saving both versions to CSV
df_final.to_csv('data/Anovator_Final_Cleaned_Dataset.csv', index=False)
df_ml.to_csv('data/Anovator_Processed_Modeling_Data.csv', index=False)

print("CSV files created in the data folder.")
print("Please upload 'Anovator_Processed_Modeling_Data.csv' for analysis.")

CSV files created in the data folder.
Please upload 'Anovator_Processed_Modeling_Data.csv' for analysis.
